# 06. Denoise Loop: Euler, CFG, time warp и partial sampling

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf).
<!-- study-navigation:end -->

От интеграла ODE перейдём к native denoise_loop и каждому helper; проверим convergence, число forward calls и замороженную модальность.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lab 1 Q1.1 (cells 9–13), notes §2.1 pp.7–9: Euler; Lab 2 Q2.3: поле; Lab 3 Part 2: CFG. Native denoise loop связывает эти три навыка.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='06_denoise_loop_euler_cfg_and_partial_sampling';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Дискретизация интеграла и ошибка

$$
x(\sigma_{k+1})=x(\sigma_k)+\int_{\sigma_k}^{\sigma_{k+1}}u(x(s),s)\,ds
\approx x_k+\Delta\sigma_k u(x_k,\sigma_k).
$$

При гладком Lipschitz field local truncation error O(h²), global O(h) на конечном интервале. Neural approximation error — отдельная величина: уменьшение h не исправляет неверное поле. Taylor expansion exact solution:

$$
x(t+h)=x(t)+hu+\frac{h^2}{2}(\partial_tu+J_u u)+O(h^3).
$$

Euler отбрасывает h² term. При bound C на него и Lipschitz constant L ошибка удовлетворяет e_(k+1)≤(1+L|h|)e_k+C h². Суммирование по конечному interval даёт global O(max|h|); отрицательный denoising step не меняет порядок оценки.

В source schedule raw τ от 1 к 0 отображается σ=aτ/(1+(a−1)τ). Network time=1000σ. Euler использует **difference shifted σ**, не raw Δτ. Здесь scheduler_scale меняет сетку, а не само Gθ как функцию σ.


In [ ]:
loop=native('core.algo.denoise_loop');schedule=native('core.algo.flow_matching')
A=torch.tensor([[.7,.2],[.3,.4]]);start=scalar_bundle();x1=vector_from_bundle(start)
exact=torch.matrix_exp(-A)@x1;rows=[]
for steps in [4,8,16,32,64]:
    field=CountingLinearField(A)
    result=loop.denoise_loop(clone_bundle(start),field,dummy_text(),dummy_text(positive=0.),None,None,
      torch.zeros(3,1,4,2,2),torch.zeros(3,1,4,2,2),steps,1.,1.)
    error=float((vector_from_bundle(result)-exact).norm())
    assert field.calls==steps
    rows.append({'steps':steps,'NFE':field.calls,'error':error})
assert rows[-1]['error']<rows[0]['error']/8
metrics['native_euler_convergence']=rows
fig,ax=plt.subplots(figsize=(6,3.5));ax.loglog([r['steps'] for r in rows],[r['error'] for r in rows],'o-',label='Native Euler vs matrix exponential');ax.loglog([4,64],[rows[0]['error'],rows[0]['error']/16],'--',label='1/N guide')
ax.set(xlabel='Network evaluations',ylabel='Endpoint L2 error');ax.legend();fig.tight_layout();fig.savefig(figure_path(LAB,'euler_convergence'));plt.show()


## 2. CFG — две оценки поля, одна Euler траектория

$$
u_w=u_{null}+w(u_{cond}-u_{null}).
$$

w=1 возвращает conditional velocity, поэтому source не вызывает null branch повторно (с допуском 1e−6). При w≠1 обычно два network forwards на один solver step. Это не два Euler шага. Negative-prompt baseline не обязательно truly unconditional model branch.

Для tiny analytic field положительное/нулевое text condition меняет постоянный bias. Проверим NFE отдельно от num_steps. Обычный CFG sweep не следует применять к distilled π checkpoint: там guided teacher behaviour уже могло быть distilled; native π sampler вообще не принимает CFG argument.


In [ ]:
rows=[]
for w in [0.,1.,3.]:
    field=CountingLinearField(torch.zeros(2,2),text_bias=.2)
    out=loop.denoise_loop(clone_bundle(start),field,dummy_text(positive=1.),dummy_text(positive=0.),None,None,
      torch.zeros(3,1,4,2,2),torch.zeros(3,1,4,2,2),8,w,1.)
    expected=x1-.2*w
    assert torch.allclose(vector_from_bundle(out),expected,atol=2e-6)
    assert field.calls==(8 if w==1 else 16)
    rows.append({'CFG':w,'steps':8,'NFE':field.calls,'endpoint_error':float((vector_from_bundle(out)-expected).abs().max())})
metrics['CFG_counts']=rows;print(rows)


## 3. Partial sampling: modality остаётся в input

Если sample_audio=False, source не обновляет audio и передаёт audio time=0; video time меняется. Это условная генерация video на уже clean audio state. Удалить audio tensor — другая branch forward; заменить его noise и объявить clean — неверный conditioning protocol.

Проверим неизменность tensor и фактические time arguments. Оба потока остаются доступны oracle; настоящий multimodal DiT также может вычислять оба streams, несмотря на отключённый solver update.


In [ ]:
field=CountingLinearField(A);before=start.audio.clone()
out=loop.denoise_loop(clone_bundle(start),field,dummy_text(),dummy_text(positive=0.),None,None,
 torch.zeros(3,1,4,2,2),torch.zeros(3,1,4,2,2),8,1.,3.,sample_audio=False)
assert torch.equal(out.audio,before)
assert all(t[1]==[0.] for t in field.time_log)
metrics['partial_sampling']={'audio_unchanged':True,'audio_time_always_zero':True,'network_calls':field.calls}
show_source('core/algo/denoise_loop.py','denoise_loop')


## 4. Helpers, RoPE caching и настоящее tiny forward

`_raw_dit` разворачивает cache wrapper только при наличии `.module` и `.set_cache`; не любую обёртку с `.module`. `_rebuild_text_rope` воспроизводит tables подходящей длины. `recompute_ropes_each_step` — диагностический A/B, он не меняет intended coordinates.

Сравним два native tiny прогонов на одном noise: precomputed RoPE и recomputation. Исходники network/loop настоящие, random weights уменьшены. Измеряем exact/numeric parity и NFE; не делаем вывода о perceptual quality.


In [ ]:
from types import SimpleNamespace
model=tiny_model().eval();v,a,kw=tiny_inputs(model)
wrapped=SimpleNamespace(module=model,set_cache=lambda:None)
assert loop._raw_dit(wrapped) is model
rebuilt=loop._rebuild_text_rope(model,kw['text_rope'])
assert all(torch.equal(x,y) for x,y in zip(rebuilt,kw['text_rope']))
B=native('core.types').LatentBundle
bundle=B(v.clone(),a.clone(),torch.tensor([0,2],dtype=torch.int32),torch.tensor([0,6],dtype=torch.int32))
def run(recompute):
    model.clear_text_proj_cache();calls=[];h=model.register_forward_hook(lambda *args:calls.append(1))
    out=loop.denoise_loop(clone_bundle(bundle),model,{'text_embeds':kw['text_embed'],'pooled_embed':kw['pooled_text_embed']},
      dummy_text(positive=0.),kw['visual_rope'],kw['audio_rope'],kw['text_rope'],kw['text_rope'],
      3,1.,3.,attention_mask=kw['attention_mask'],recompute_ropes_each_step=recompute)
    h.remove();return out,len(calls)
base,n=run(False);recomputed,m=run(True)
err=max(float((base.video-recomputed.video).abs().max()),float((base.audio-recomputed.audio).abs().max()))
assert err<2e-6 and n==m==3
metrics['native_tiny_rope_AB']={'max_error':err,'NFE_each':n}


## Измерение → вывод

Native Euler дал ожидаемое уменьшение ошибки и точные CFG branch counts. Partial modality tensor сохранился, frozen time был нулевым. Recomputed/precomputed RoPE дали совпадающий tiny result. Ни network error, ни production speed этим не измерены.


## Самостоятельные опыты

1. Выполните MIT Lab 1 Q1.1 и сравните convergence с нашим oracle.
2. При scheduler_scale=5 повторите convergence; сравните Δσ, а не raw Δτ.
3. Запустите sample_video=False и объясните роли data/noise в frozen video.
4. Добавьте first-frame reference и проверьте constraint после последнего update.

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
